In [2]:
# 1. Imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_text_splitters import CharacterTextSplitter

import os
from dotenv import load_dotenv
load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")

# 2. Setup
llm = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.5-flash", google_api_key=google_api_key)

# 3. Load document and split
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents([raw_text])

# 4. Define prompts
map_prompt = PromptTemplate.from_template("Summarize:\n\n{context}")
reduce_prompt = PromptTemplate.from_template("Combine the following summaries into a single answer:\n\n{context}")

# 5. Create map + reduce chains
map_chain = map_prompt | llm
reduce_chain = reduce_prompt | llm

# 6. Use RunnableLambda to convert documents
def map_docs(docs):
    return [{"context": doc.page_content} for doc in docs]



def _normalize_content(content):
    if isinstance(content, list):
        return "\n".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    return content

def extract_texts(responses):
    return {"context": "\n".join(_normalize_content(res.content) for res in responses)}

map_reduce_chain = (
    RunnableLambda(map_docs)
    | map_chain.map()
    | RunnableLambda(extract_texts)
    | reduce_chain
)

result = map_reduce_chain.invoke(docs)
print("📄 Final Summary:\n", _normalize_content(result.content))

📄 Final Summary:
 Created by Harrison Chase in October 2022, LangChain is a widely used framework for building LLM-powered applications, such as chatbots and autonomous agents. It facilitates the development of these applications by enabling model chaining, RAG (Retrieval-Augmented Generation) integration, vector store connectivity, and the creation of tool-using agents.
